i have:

- libristutter stuttered audio
- melotts good audio

In [1]:
import librosa
import matplotlib.pyplot as plt
import librosa.display
from scipy.io.wavfile import write
import os
import numpy as np
from tensorflow.keras import layers
from IPython.display import Audio
import wave
import pandas as pd
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt

2024-12-27 21:06:14.385632: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1735351574.396699   16426 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1735351574.399902   16426 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-12-27 21:06:14.412733: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
fp = "/home/alien/Git/XSpeech/data_processing/output.csv"

In [3]:
df = pd.read_csv(fp)
df

,filepath,results,new_filepath
0,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...
1,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...
2,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...
3,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...
4,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...
...,...,...,...
3908,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...
3909,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...
3910,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...
3911,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/LibriStutterData/LibriStu...,/home/alien/Git/DATA/MeloTTSAudioLibriStutter/...


In [4]:
# Load a .wav file
def load_wav(filename, sr=44100):
    y, sr = librosa.load(filename, sr=sr)
    return y, sr

In [5]:
# i dont know what im doing
# Convert the audio to a spectrogram (e.g., Mel-spectrogram)
def audio_to_mel_spec(y, sr=44100, n_fft=2048, hop_length=512, n_mels=128):
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)  # Convert power to decibels
    return mel_spec_db

# Convert the mel-spectrogram back to a waveform
def mel_spec_to_audio(mel_spec_db, sr=44100, n_fft=2048, hop_length=512):
    mel_spec = librosa.db_to_power(mel_spec_db)  # Convert back to power
    y = librosa.feature.inverse.mel_to_audio(mel_spec, sr=sr, n_fft=n_fft, hop_length=hop_length)
    return y

In [6]:
y, sr = load_wav('/home/alien/Git/DATA/MeloTTSAudioLibriStutter/103-1240-0000.wav')
y

array([-3.0517578e-05, -3.0517578e-05,  0.0000000e+00, ...,
        0.0000000e+00,  0.0000000e+00,  0.0000000e+00], dtype=float32)

In [7]:
mel_spec_db = audio_to_mel_spec(y)
mel_spec_db

array([[-80., -80., -80., ..., -80., -80., -80.],
       [-80., -80., -80., ..., -80., -80., -80.],
       [-80., -80., -80., ..., -80., -80., -80.],
       ...,
       [-80., -80., -80., ..., -80., -80., -80.],
       [-80., -80., -80., ..., -80., -80., -80.],
       [-80., -80., -80., ..., -80., -80., -80.]], dtype=float32)

In [8]:
y_new = mel_spec_to_audio(mel_spec_db)
y_new

array([-8.3957311e-06,  4.5383213e-06,  1.5231709e-05, ...,
       -1.4699754e-05, -1.0827908e-05,  4.7024482e-06], dtype=float32)

In [9]:
def save_audio_to_wav(y, sr, filename):
    """
    Save the audio signal 'y' as a .wav file.

    Parameters:
    y (numpy array): The audio signal (should be 1D).
    sr (int): The sampling rate (samples per second).
    filename (str): The path to the output .wav file.
    """
    # Ensure the audio signal is in the correct format (16-bit PCM)
    # Normalize the audio signal to fit within the 16-bit range
    y_normalized = np.int16(y / np.max(np.abs(y)) * 32767)

    # Write the audio data to a .wav file
    write(filename, sr, y_normalized)

In [10]:
save_audio_to_wav(y_new, 44100, "output.wav")

In [10]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor

model = WhisperForConditionalGeneration.from_pretrained("justanotherinternetguy/whisper-small-sep28")
processor = WhisperProcessor.from_pretrained("justanotherinternetguy/whisper-small-sep28")

OSError: Can't load tokenizer for 'justanotherinternetguy/whisper-small-sep28'. If you were trying to load it from 'https://huggingface.co/models', make sure you don't have a local directory with the same name. Otherwise, make sure 'justanotherinternetguy/whisper-small-sep28' is the correct path to a directory containing all relevant files for a WhisperTokenizer tokenizer.

In [3]:
from transformers import pipeline

# Load the model for ASR
pipe = pipeline(model="justanotherinternetguy/whisper-small-sep28")

def transcribe(audio):
    # Process the audio file (assumes it's a .wav file or other audio format Whisper supports)
    text = pipe(audio)["text"]
    return text

print(transcribe('/home/alien/Git/DATA/MeloTTSAudioLibriStutter/103-1240-0000.wav'))

Device set to use cuda:0
/home/alien/Programming/env/lib/python3.12/site-packages/transformers/models/whisper/generation_whisper.py:512: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, 50259], [2, 50359], [3, 50363]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.


 Chapter 1 of The Cessrachalind is surprised Mrs. Rachel Lind lived just where the avenue and main road dipped down into a little halo fringed with alders and ladies' eardrops and produced by a brook.
